<a href="https://colab.research.google.com/github/pratchayapron/229351-StatisticalLearning-or-Statistical-Learning-Labs/blob/main/Lab04_Naive_Bayes_Grid_and_Random_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Statistical Learning for Data Science 2 (229352)
#### Instructor: Donlapark Ponnoprat

#### [Course website](https://donlapark.pages.dev/229352/)

## Lab #4

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

from scipy.stats import uniform

In [ ]:
train = fetch_20newsgroups(subset='train')
test = fetch_20newsgroups(subset='test')

Xtrain = train.data[:3000]
ytrain = train.target[:3000]
Xtest = test.data[:500]
ytest = test.target[:500]

print("X:", len(Xtest))
print("y:", len(ytest))

X: 500
y: 500


### Naive Bayes [(Documentation)](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB(alpha=0.1)

### Random Search Cross-Validation [(Documentation)](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)

### Uniform distribution in `Scipy` [(Documentation)](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.uniform.html)

In [ ]:
pipeline = Pipeline([('tfidf' ,TfidfVectorizer()), ('nb', MultinomialNB())])
parameter = {'nb__alpha' : uniform(loc=0, scale=10)}
clf = RandomizedSearchCV(pipeline, parameter, n_iter=7)
clf.fit(Xtrain, ytrain)

RandomizedSearchCV(estimator=Pipeline(steps=[('tfidf', TfidfVectorizer()),
                                             ('nb', MultinomialNB())]),
                   n_iter=7,
                   param_distributions={'nb__alpha': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7e096544deb0>})

#### Exercise

1. For the Naive Bayes model, use grid search 5-fold cross-validation across different values of `alpha` to find the best model.

2. For the best value of `alpha`, compute the `f1_macro` score on the test set.
* What value of `alpha` did you obtain?
* What is the model's `f1_macro` score?

3. Repeat Exercise 1 and 2 for **random search** 5-fold cross validation across different values of `alpha`. Compute the `f1_macro` score on the test set.
* What value of `alpha` did you obtain?
* Did you get a better `f1_macro` score compared to grid search in Exercise 2?

In [ ]:
print("=" * 60)
print("GRID SEARCH CROSS-VALIDATION")
print("=" * 60)

# Create pipeline
pipeline_grid = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB())
])


param_grid = {
    'nb__alpha': [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
}


grid_search = GridSearchCV(
    pipeline_grid,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

print("Fitting Grid Search...")
grid_search.fit(Xtrain, ytrain)


best_alpha_grid = grid_search.best_params_['nb__alpha']
print(f"\nBest alpha (Grid Search): {best_alpha_grid}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

# Predict on test set
y_pred_grid = grid_search.predict(Xtest)
f1_macro_grid = f1_score(ytest, y_pred_grid, average='macro')

print(f"Test F1-macro score: {f1_macro_grid:.4f}")
print()

GRID SEARCH CROSS-VALIDATION
Fitting Grid Search...
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best alpha (Grid Search): 0.001
Best CV score: 0.8316
Test F1-macro score: 0.7442



In [ ]:
print("=" * 60)
print("RANDOM SEARCH CROSS-VALIDATION")
print("=" * 60)

# Create pipeline
pipeline_random = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

param_dist = {
    'nb__alpha': uniform(loc=0.001, scale=10)
}

random_search = RandomizedSearchCV(
    pipeline_random,
    param_dist,
    n_iter=20,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

print("Fitting Random Search...")
random_search.fit(Xtrain, ytrain)


best_alpha_random = random_search.best_params_['nb__alpha']
print(f"\nBest alpha (Random Search): {best_alpha_random:.4f}")
print(f"Best CV score: {random_search.best_score_:.4f}")

# Predict on test set
y_pred_random = random_search.predict(Xtest)
f1_macro_random = f1_score(ytest, y_pred_random, average='macro')

print(f"Test F1-macro score: {f1_macro_random:.4f}")
print()

RANDOM SEARCH CROSS-VALIDATION
Fitting Random Search...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best alpha (Random Search): 0.2068
Best CV score: 0.7620
Test F1-macro score: 0.6862



In [ ]:
print("=" * 60)
print("COMPARISON")
print("=" * 60)
print(f"Grid Search:")
print(f"  Best alpha: {best_alpha_grid}")
print(f"  Test F1-macro: {f1_macro_grid:.4f}")
print()
print(f"Random Search:")
print(f"  Best alpha: {best_alpha_random:.4f}")
print(f"  Test F1-macro: {f1_macro_random:.4f}")
print()
print(f"Improvement: {f1_macro_random - f1_macro_grid:+.4f}")
print(f"Random Search is {'BETTER' if f1_macro_random > f1_macro_grid else 'WORSE'}")
print()


print("=" * 60)
print("DETAILED CLASSIFICATION REPORT (Best Model)")
print("=" * 60)
best_model = random_search if f1_macro_random > f1_macro_grid else grid_search
y_pred_best = best_model.predict(Xtest)
print(classification_report(ytest, y_pred_best, target_names=train.target_names))

COMPARISON
Grid Search:
  Best alpha: 0.001
  Test F1-macro: 0.7442

Random Search:
  Best alpha: 0.2068
  Test F1-macro: 0.6862

Improvement: -0.0580
Random Search is WORSE

DETAILED CLASSIFICATION REPORT (Best Model)
                          precision    recall  f1-score   support

             alt.atheism       0.81      0.62      0.70        21
           comp.graphics       0.52      0.52      0.52        21
 comp.os.ms-windows.misc       0.65      0.58      0.61        26
comp.sys.ibm.pc.hardware       0.64      0.68      0.66        34
   comp.sys.mac.hardware       0.83      0.71      0.76        34
          comp.windows.x       0.83      0.58      0.68        26
            misc.forsale       0.82      0.82      0.82        22
               rec.autos       0.82      0.96      0.89        28
         rec.motorcycles       0.93      0.85      0.89        33
      rec.sport.baseball       0.88      0.88      0.88        25
        rec.sport.hockey       0.89      0.93      0.9